In [22]:
!rm -r "GalRotpy"
!git clone https://github.com/andresGranadosC/GalRotpy.git --branch master
!pip3 install -r GalRotpy/requirements.txt >> log_req.txt

Cloning into 'GalRotpy'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 224 (delta 58), reused 82 (delta 30), pack-reused 73 (from 1)
Receiving objects: 100% (224/224), 4.25 MiB | 20.74 MiB/s, done.
Resolving deltas: 100% (86/86), done.


# Importar paquetes de Python
## necesarios para cálculos y visualización

In [23]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# from matplotlib.widgets import Slider, Button, RadioButtons, CheckButtons, TextBox # Matplotlib widgets
import matplotlib.pylab as plt # Plotting interface
import numpy as np
from galpy.potential import MiyamotoNagaiPotential, NFWPotential, RazorThinExponentialDiskPotential, BurkertPotential # GALPY potentials
from galpy.potential import calcRotcurve # composed rotation curve calculation for plotting
from astropy import units # Physical/real units data managing
from astropy import table as Table # For fast and easy reading / writing with tables using numpy library
import emcee
import corner
import time
import re
import pandas as pd
import multiprocessing as mp
from scipy.optimize import fsolve
from scipy.optimize import curve_fit

import ipywidgets as widgets

import base64
import io
from PIL import Image


# Leer los documentos de texto:
## Curva de rotación de la galaxia y
## Datos de parámetros físicos de la galaxia

In [25]:
init_guess_params = Table.Table.read('GalRotpy/M33_guess_params.txt', format='ascii.tab')

c_bulge, amp1, a1, b1, include_bulge = init_guess_params[0]
c_tn, amp2, a2, b2, include_tn = init_guess_params[1]
c_tk, amp3, a3, b3, include_tk = init_guess_params[2]
c_ex, amp4, h_r, vertical_ex, include_ex = init_guess_params[3]
c_dh, amp5, a5, b5, include_dh = init_guess_params[4]
c_bh, amp6, a6, b6, include_bh = init_guess_params[5]

tt=Table.Table.read('GalRotpy/M33_rot_curve.txt', format='ascii.tab') # Rotation curve
x_offset = 0.0  # It defines a radial coordinate offset as user input
r_0=1*units.kpc # units
v_0=220*units.km/units.s # units
# Real data:
r_data=tt['r']-x_offset # The txt file must contain the radial coordinate values in kpc
v_c_data=tt['vel'] # velocity in km/s
v_c_err_data = tt['e_vel'] # and velocity error in km/s

# This loop is needed since galpy fails when r=0 or very close to 0
for i in range(len(r_data)):
    if r_data[i]<1e-3:
        r_data[i]=1e-3

lista=np.linspace(0.001, 1.02*np.max(r_data), 10*len(r_data)) # radial coordinate for the rotation curve calculation

class MiyamotoNagaiP:
    def __init__(self, dict_params):
        self.amp = dict_params['amp']
        self.a = dict_params['a']
        self.b = dict_params['b']

    def __str__(self):
        return f"amp={self.amp}, a={self.a}, b={self.b}"

    def __repr__(self):
        return f"amp={self.amp}, a={self.a}, b={self.b}"

class MassScaleP:
    def __init__(self, dict_params):
        self.amp = dict_params['amp']
        self.a = dict_params['a']

    def __str__(self):
        return f"amp={self.amp}, a={self.a}"

    def __repr__(self):
        return f"amp={self.amp}, a={self.a}"


# Definición de los potenciales gravitacionales

In [26]:

# Bulge
def MN_b_amp_s_func(val):
    if MN_b_plot.get_visible() == True:
        global MN_Bulge_p, amp1, a1, b1
        amp1=val*1
        MN_Bulge_p = MiyamotoNagaiPotential(amp=val*units.Msun,a=a1*units.kpc,b=b1*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_b_a_s_func(val):
    if MN_b_plot.get_visible() == True:
        global MN_Bulge_p, amp1, a1, b1
        a1=val*1
        MN_Bulge_p = MiyamotoNagaiPotential(amp=amp1*units.Msun,a=val*units.kpc,b=b1*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_b_b_s_func(val):
    if MN_b_plot.get_visible() == True:
        global MN_Bulge_p, amp1, a1, b1
        b1=val*1
        MN_Bulge_p = MiyamotoNagaiPotential(amp=amp1*units.Msun,a=a1*units.kpc,b=val*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()

# Thin disk
def MN_td_amp_s_func(val):
    if MN_td_plot.get_visible() == True:
        global MN_Thin_Disk_p, amp2, a2, b2
        amp2=val*1
        MN_Thin_Disk_p= MiyamotoNagaiPotential(amp=val*units.Msun,a=a2*units.kpc,b=b2*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_td_a_s_func(val):
    if MN_td_plot.get_visible() == True:
        global MN_Thin_Disk_p, amp2, a2, b2
        a2=val*1
        MN_Thin_Disk_p= MiyamotoNagaiPotential(amp=amp2*units.Msun,a=val*units.kpc,b=b2*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_td_b_s_func(val):
    if MN_td_plot.get_visible() == True:
        global MN_Thin_Disk_p, amp2, a2, b2
        b2=val*1
        MN_Thin_Disk_p= MiyamotoNagaiPotential(amp=amp2*units.Msun,a=a2*units.kpc,b=val*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()

# Thick disk
def MN_tkd_amp_s_func(val):
    if MN_tkd_plot.get_visible() == True:
        global MN_Thick_Disk_p, amp3, a3, b3
        amp3=val*1
        MN_Thick_Disk_p= MiyamotoNagaiPotential(amp=val*units.Msun,a=a3*units.kpc,b=b3*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_tkd_a_s_func(val):
    if MN_tkd_plot.get_visible() == True:
        global MN_Thick_Disk_p, amp3, a3, b3
        a3=val*1
        MN_Thick_Disk_p= MiyamotoNagaiPotential(amp=amp3*units.Msun,a=val*units.kpc,b=b3*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()
def MN_tkd_b_s_func(val):
    if MN_tkd_plot.get_visible() == True:
        global MN_Thick_Disk_p, amp3, a3, b3
        b3=val*1
        MN_Thick_Disk_p= MiyamotoNagaiPotential(amp=amp3*units.Msun,a=a3*units.kpc,b=val*units.kpc,normalize=False,ro=r_0, vo=v_0)
        update_rot_curve()

# Exponential disk
def MN_ed_amp_s_func(val):
    if EX_d_plot.get_visible() == True:
        global EX_Disk_p, amp4,h_r
        amp4=val*1
        EX_Disk_p = RazorThinExponentialDiskPotential(amp=val*(units.Msun/(units.pc**2)), hr=h_r*units.kpc, maxiter=20, tol=0.001, normalize=False, ro=r_0, vo=v_0, new=True, glorder=100)
        update_rot_curve()
def MN_ed_a_s_func(val):
    if EX_d_plot.get_visible() == True:
        global EX_Disk_p, amp4,h_r
        h_r=val*1
        EX_Disk_p = RazorThinExponentialDiskPotential(amp=amp4*(units.Msun/(units.pc**2)), hr=val*units.kpc, maxiter=20, tol=0.001, normalize=False, ro=r_0, vo=v_0, new=True, glorder=100)
        update_rot_curve()

# NFW Halo
def NFW_amp_s_func(val):
    if NFW_plot.get_visible() == True:
        global NFW_p, amp5,a5
        amp5=val*1
        NFW_p = NFWPotential(amp=val*units.Msun, a=a5*units.kpc, normalize=False, ro=r_0, vo=v_0)
        update_rot_curve()
def NFW_a_s_func(val):
    if NFW_plot.get_visible() == True:
        global NFW_p, amp5,a5
        a5=val*1
        NFW_p = NFWPotential(amp=amp5*units.Msun, a=val*units.kpc, normalize=False, ro=r_0, vo=v_0)
        update_rot_curve()

# Burkert Halo
def BK_amp_s_func(val):
    if BK_plot.get_visible() == True:
        global BK_p, amp6,a6
        amp6=val*1
        BK_p = BurkertPotential(amp=val*units.Msun/(units.kpc)**3, a=a6*units.kpc, normalize=False, ro=r_0, vo=v_0)
        update_rot_curve()
def BK_a_s_func(val):
    if BK_plot.get_visible() == True:
        global BK_p, amp6,a6
        a6=val*1
        BK_p = BurkertPotential(amp=amp6*units.Msun/(units.kpc)**3, a=val*units.kpc, normalize=False, ro=r_0, vo=v_0)
        update_rot_curve()

data_rows = [('BULGE', 110000000.0, 1.0, 0.0, 20, 0.495, 70),
             ('THIN DISK', 3900000000.0, 1.0, 5.3, 90, 0.25, 1),
             ('THICK DISK', 39000000000.0, 0.5, 2.6, 20, 0.8, 1),
             ('EXP DISK', 500.0, 0.5, 5.3, 90, 0.0, 0),
             ('DARK HALO', 140000000000.0, 1.0, 13.0, 90, 0.0, 0),
             ('BURKERT HALO', 8000000.0, 1.0, 20.0, 90, 0.0, 0)]
input_params = Table.Table(rows=data_rows, names=('component', 'mass', 'threshold_mass', 'a (kpc)', 'threshold_a', 'b (kpc)', 'threshold_b'))

def input_component(component, guess_mass, guess_a, guess_b):

    component_mass, component_scale_a, component_scale_b = guess_mass, guess_a, guess_b

    print('Set the guess parameters for', component)
    try:
        component_mass = float(input('Mass (in M_sun):'))
    except:
        print('No valid Mass for', component, '. It will be taken the default mass:', component_mass, 'M_sun')


    try:
        component_scale_a = float(input('Radial Scale Length (in kpc):'))
    except:
        print('No valid Radial Scale Length for', component, '. It will be taken the default Radial Scale Lenght:', component_scale_a, 'kpc')

    if component not in ['EXP DISK', 'DARK HALO', 'BURKERT HALO' ]:
        try:
            component_scale_b = float(input('Vertical Scale Length (in kpc):'))
        except:
            print('No valid Vertical Scale Length for', component, '. It will be taken the default Vertical Scale Lenght:', component_scale_b, 'kpc')

    return component_mass, component_scale_a, component_scale_b

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

x_offset = 0.0  # It defines a radial coordinate offset as user input
r_0=1*units.kpc # units
v_0=220*units.km/units.s # units

# Real data:
r_data=tt['r']-x_offset # The txt file must contain the radial coordinate values in kpc
v_c_data=tt['vel'] # velocity in km/s
v_c_err_data = tt['e_vel'] # and velocity error in km/s

# This loop is needed since galpy fails when r=0 or very close to 0
for i in range(len(r_data)):
    if r_data[i]<1e-3:
        r_data[i]=1e-3

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Initial parameters:

c_bulge, amp1, delta_mass_bulge, a1, delta_radial_bulge, b1, delta_vertical_bulge = input_params[0]
c_tn, amp2, delta_mass_tn, a2, delta_radial_tn, b2, delta_vertical_tn = input_params[1]
c_tk, amp3, delta_mass_tk, a3, delta_radial_tk, b3, delta_vertical_tk = input_params[2]
c_ex, amp4, delta_mass_ex, h_r, delta_radial_ex, vertical_ex, delta_vertical_ex = input_params[3]
c_dh, amp5, delta_mass_dh, a5, delta_radial_dh, b5, delta_vertical_dh = input_params[4]
c_bh, amp6, delta_mass_bh, a6, delta_radial_bh, b6, delta_vertical_bh = input_params[5]

components = {
    'MN_b':   ('MN Bulge',      lambda: MN_Bulge_p,      'gray',   lambda: 2 if a1 == 0. else 3),
    'MN_td':  ('MN Thin Disk',  lambda: MN_Thin_Disk_p,  'purple', lambda: 3),
    'MN_tkd': ('MN Thick Disk', lambda: MN_Thick_Disk_p, 'blue',   lambda: 3),
    'NFW':    ('NFW',           lambda: NFW_p,           'green',  lambda: 2),
    'EX_d':   ('EX Disk',       lambda: EX_Disk_p,       'cyan',   lambda: 2),
    'BK':     ('BK',            lambda: BK_p,            'orange', lambda: 2),
}

def mass_slider(label, val, lo, hi, color):
    """Log slider: masses/densities span orders of magnitude."""
    return widgets.FloatLogSlider(
        value=val, base=10, min=np.log10(lo), max=np.log10(hi), step=0.01,
        description=label, readout_format='.1e', continuous_update=False,
        style={'handle_color': color, 'description_width': '90px'},
        layout=widgets.Layout(width='380px'))

def lin_slider(label, val, lo, hi, color):
    return widgets.FloatSlider(
        value=val, min=lo, max=hi, step=(hi - lo) / 200,
        description=label, readout_format='.1f', continuous_update=False,
        style={'handle_color': color, 'description_width': '90px'},
        layout=widgets.Layout(width='380px'))

def checkbox(label):
    return widgets.Checkbox(value=True, description=label, indent=False)

W = {}  # every widget, keyed by the name update() receives

# Bulge - gray
W['vis_MN_b'] = checkbox('MN Bulge')
W['amp1'] = mass_slider(r"$M\ (M_\odot)$", amp1, amp1/10**delta_mass_bulge, amp1*10**delta_mass_bulge, 'gray')
W['a1']   = lin_slider(r"$a\ (kpc)$", a1, 0, 0.1*delta_radial_bulge, 'gray')
W['b1']   = lin_slider(r"$b\ (kpc)$", b1, b1*(1-0.01*delta_vertical_bulge), b1*(1+0.01*delta_vertical_bulge), 'gray')

# Thin disk - purple
W['vis_MN_td'] = checkbox('MN Thin Disk')
W['amp2'] = mass_slider(r"$M\ (M_\odot)$", amp2, amp2/10**delta_mass_tn, amp2*10**delta_mass_tn, 'purple')
W['a2']   = lin_slider(r"$a\ (kpc)$", a2, a2*(1-0.01*delta_radial_tn), a2*(1+0.01*delta_radial_tn), 'purple')
W['b2']   = mass_slider(r"$b\ (kpc)$", b2, b2/10**delta_vertical_tn, b2*10**delta_vertical_tn, 'purple')

# Thick disk - blue
W['vis_MN_tkd'] = checkbox('MN Thick Disk')
W['amp3'] = mass_slider(r"$M\ (M_\odot)$", amp3, amp3/10**delta_mass_tk, amp3*10**delta_mass_tk, 'blue')
W['a3']   = lin_slider(r"$a\ (kpc)$", a3, a3*(1-0.01*delta_radial_tk), a3*(1+0.01*delta_radial_tk), 'blue')
W['b3']   = mass_slider(r"$b\ (kpc)$", b3, b3/10**delta_vertical_tk, b3*10**delta_vertical_tk, 'blue')

# Exponential disk - cyan
W['vis_EX_d'] = checkbox('EX Disk')
W['amp4'] = mass_slider(r"$\Sigma_0\ (M_\odot/pc^2)$", amp4, amp4/10**delta_mass_ex, amp4*10**delta_mass_ex, 'cyan')
W['h_r']  = lin_slider(r"$h_r\ (kpc)$", h_r, h_r*(1-0.01*delta_radial_ex), h_r*(1+0.01*delta_radial_ex), 'cyan')

# NFW halo - green
W['vis_NFW'] = checkbox('NFW')
W['amp5'] = mass_slider(r"$M_0\ (M_\odot)$", amp5, amp5/10**delta_mass_dh, amp5*10**delta_mass_dh, 'green')
W['a5']   = lin_slider(r"$a\ (kpc)$", a5, a5*(1-0.01*delta_radial_dh), a5*(1+0.01*delta_radial_dh), 'green')

# Burkert halo - orange
W['vis_BK'] = checkbox('BK')
W['amp6'] = mass_slider(r"$\rho_0\ (M_\odot/kpc^3)$", amp6, amp6/10**delta_mass_bh, amp6*10**delta_mass_bh, 'orange')
W['a6']   = lin_slider(r"$a\ (kpc)$", a6, a6*(1-0.01*delta_radial_bh), a6*(1+0.01*delta_radial_bh), 'orange')

# ---------- Potential builders (replace the *_s_func callbacks) ----------
def build_MN_b(p):
    return MiyamotoNagaiPotential(amp=p['amp1']*units.Msun, a=p['a1']*units.kpc, b=p['b1']*units.kpc,
                                  normalize=False, ro=r_0, vo=v_0)
def build_MN_td(p):
    return MiyamotoNagaiPotential(amp=p['amp2']*units.Msun, a=p['a2']*units.kpc, b=p['b2']*units.kpc,
                                  normalize=False, ro=r_0, vo=v_0)
def build_MN_tkd(p):
    return MiyamotoNagaiPotential(amp=p['amp3']*units.Msun, a=p['a3']*units.kpc, b=p['b3']*units.kpc,
                                  normalize=False, ro=r_0, vo=v_0)
def build_EX_d(p):
    return RazorThinExponentialDiskPotential(amp=p['amp4']*(units.Msun/units.pc**2), hr=p['h_r']*units.kpc,
                                             normalize=False, ro=r_0, vo=v_0,
                                             new=True, glorder=100)
def build_NFW(p):
    return NFWPotential(amp=p['amp5']*units.Msun, a=p['a5']*units.kpc, normalize=False, ro=r_0, vo=v_0)
def build_BK(p):
    return BurkertPotential(amp=p['amp6']*units.Msun/units.kpc**3, a=p['a6']*units.kpc,
                            normalize=False, ro=r_0, vo=v_0)

# key: (label, builder, color, N-rule)  -- same plotting order as the original code
components = {
    'MN_b':   ('MN Bulge',      build_MN_b,   'gray',   lambda p: 2 if p['a1'] == 0. else 3),
    'MN_td':  ('MN Thin Disk',  build_MN_td,  'purple', lambda p: 3),
    'MN_tkd': ('MN Thick Disk', build_MN_tkd, 'blue',   lambda p: 3),
    'NFW':    ('NFW',           build_NFW,    'green',  lambda p: 2),
    'EX_d':   ('EX Disk',       build_EX_d,   'cyan',   lambda p: 2),
    'BK':     ('BK',            build_BK,     'orange', lambda p: 2),
}

# Results from the latest update, readable from other cells
state = {'N': [], 'composite_pot_array': [], 'potentials': {}}

def update(**p):
    N, composite_pot_array = [], []

    fig, ax = plt.subplots(figsize=(8, 6))
    # ax.plot(lista, v_c_data, 'k.', label='data')
    CV_galaxy = ax.errorbar(r_data - x_offset, v_c_data, v_c_err_data,  c='k', fmt='', ls='none')
    CV_galaxy_dot = ax.scatter(r_data - x_offset, v_c_data, c='k')

    for key, (label, build, color, get_N) in components.items():
        if not p[f'vis_{key}']:
            continue
        pot = build(p)
        N.append(get_N(p))
        curve = calcRotcurve(pot, lista, phi=None) * 220
        ax.plot(lista, curve, linestyle='--', c=color, label=label)
        composite_pot_array.append(pot)
        state['potentials'][key] = pot

    v_circ_comp = calcRotcurve(composite_pot_array, lista, phi=None)*220
    v_circ_comp_plot, = ax.plot(lista, v_circ_comp, c='k')
    ax.set_xlabel(r'$R\,(\mathrm{kpc})$', fontsize=20)
    ax.set_ylabel(r'$v_c\,(\mathrm{km/s})$', fontsize=20)
    ax.tick_params(axis='both', which='both', labelsize=15)
    ax.set_xlim([0, np.max(lista)])
    ax.set_ylim([0, np.max(v_c_data) * 1.2])
    ax.legend(fontsize=12)
    plt.show()

    state['N'] = N
    state['composite_pot_array'] = composite_pot_array

out = widgets.interactive_output(update, W)

# ---------- Layout: one column per component ----------
def column(key, params):
    return widgets.VBox([W[f'vis_{key}']] + [W[k] for k in params])

controls = widgets.VBox([
    column('MN_b',   ['amp1', 'a1', 'b1']),
    column('MN_td',  ['amp2', 'a2', 'b2']),
    column('MN_tkd', ['amp3', 'a3', 'b3']),
    column('EX_d',   ['amp4', 'h_r']),
    column('NFW',    ['amp5', 'a5']),
    column('BK',     ['amp6', 'a6']),
])

def parse_potential(pot_list):
    # Split into one string per potential (each starts with "<Name>Potential with")
    # entries = re.findall(r"\w+Potential with .*?(?=,\s*\w+Potential with|\]\s*$)", pot_list, flags=re.S)

    rows = []
    for s in pot_list:
        """Parse one galpy potential string into a dict."""
        s = str(s).strip().strip("[]").strip()
        row = {"potential": s.split(" with ")[0]}

        params = re.search(r"internal parameters:\s*(.*?)\s+and physical outputs", s, flags=re.S)
        if params:
            for item in params.group(1).split(","):
                key, val = [x.strip() for x in item.split("=")]
                if val in ("True", "False"):
                    row[key] = val == "True"
                else:
                    try:
                        row[key] = float(val)
                    except ValueError:
                        row[key] = val

        ro = re.search(r"ro=([\d.eE+-]+)\s*kpc", s)
        vo = re.search(r"vo=([\d.eE+-]+)\s*km/s", s)
        row["ro_kpc"] = float(ro.group(1)) if ro else None
        row["vo_kms"] = float(vo.group(1)) if vo else None
        rows.append(row)
    return rows

def save_output_to_pdf(out, filename="Rotation_curve.pdf"):
    images = []
    for item in out.outputs:
        data = item.get("data", {}) if item.get("output_type") == "display_data" else {}
        if "image/png" in data:
            png_bytes = base64.b64decode(data["image/png"])
            images.append(Image.open(io.BytesIO(png_bytes)).convert("RGB"))

    if not images:
        raise ValueError("No PNG image found in the Output widget.")

    # Saves all images as pages in a single PDF
    images[0].save(filename, save_all=True, append_images=images[1:])




# Inicio

In [27]:
display(widgets.HBox([controls, out]))

# Luego de ajustar los parametros ejecuta estas siguientes lineas

In [28]:
rows = parse_potential(state['composite_pot_array'])
df = pd.DataFrame(rows)
df.to_csv("Galaxy_parameters.csv", index=False)

# Reconstruct and save the plot directly to PDF using matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(r_data - x_offset, v_c_data, v_c_err_data, c='k', fmt='', ls='none')
ax.scatter(r_data - x_offset, v_c_data, c='k')

for key, (label, build, color, _) in components.items():
    # Re-draw each component that was saved in the state potentials dict
    if key in state['potentials']:
        pot = state['potentials'][key]
        curve = calcRotcurve(pot, lista, phi=None) * 220
        ax.plot(lista, curve, linestyle='--', c=color, label=label)

if state['composite_pot_array']:
    v_circ_comp = calcRotcurve(state['composite_pot_array'], lista, phi=None) * 220
    ax.plot(lista, v_circ_comp, c='k')

ax.set_xlabel(r'$R\,(\mathrm{kpc})$', fontsize=20)
ax.set_ylabel(r'$v_c\,(\mathrm{km/s})$', fontsize=20)
ax.tick_params(axis='both', which='both', labelsize=15)
ax.set_xlim([0, np.max(lista)])
ax.set_ylim([0, np.max(v_c_data) * 1.2])
ax.legend(fontsize=12)

# Save the reconstructed matplotlib figure
plt.savefig("Rotation_curve.pdf", format="pdf", bbox_inches="tight")
plt.close(fig)
print("Saved Galaxy_parameters.csv and Rotation_curve.pdf successfully!")

Saved Galaxy_parameters.csv and Rotation_curve.pdf successfully!
